# Used Car Price Prediction — Final Notebook
## End-to-End Machine Learning Project

**Objective:** predict the selling price of a used vehicle from its year, present price, mileage, fuel type, selling type, transmission, and ownership information.

This notebook follows an end-to-end data-mining workflow: business understanding, data understanding, data preparation, modeling, evaluation, and deployment planning. This structure is consistent with the CRISP-DM lifecycle. [IBM CRISP-DM](https://www.ibm.com/docs/en/spss-modeler/18.6.0?topic=dm-crisp-help-overview)


## 1. Business Understanding
A used-car marketplace can use a price prediction model to estimate a reasonable selling price from vehicle attributes. The target variable is `Selling_Price`. The model is intended as a decision-support estimate, not an automatic pricing authority.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

df = pd.read_csv("../data/raw/car data.csv")
df.head()

## 2. Data Understanding

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)
print("Missing values:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Target distribution
plt.figure(figsize=(8,5))
plt.hist(df["Selling_Price"], bins=30)
plt.title("Selling Price Distribution")
plt.xlabel("Selling Price")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["Present_Price"], df["Selling_Price"], alpha=.65)
plt.title("Present Price vs Selling Price")
plt.xlabel("Present Price")
plt.ylabel("Selling Price")
plt.show()

## 3. Data Preparation
There are no missing cells. Two exact duplicate rows are removed. `Car_Name` is retained in the source data but excluded from the baseline predictive features because it is a high-cardinality name field and can encourage memorization rather than generalizable pricing behavior. `Car_Age` is engineered relative to the maximum observed year in this dataset.

In [ ]:
df = df.drop_duplicates().copy()
df["Car_Age"] = df["Year"].max() - df["Year"]

target = "Selling_Price"
features = ["Year","Present_Price","Driven_kms","Fuel_Type","Selling_type","Transmission","Owner","Car_Age"]

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_features = ["Year","Present_Price","Driven_kms","Owner","Car_Age"]
categorical_features = ["Fuel_Type","Selling_type","Transmission"]

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

## 4. Modeling
Four regression baselines are compared: Linear Regression, Random Forest, Extra Trees, and Gradient Boosting. Model selection considers both a held-out test set and 5-fold cross-validation; the cross-validation result is used to reduce the chance of selecting a model based only on one split.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=500, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=500, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42, n_estimators=300, max_depth=2,
        learning_rate=0.04, loss="huber"
    )
}

results = []
pipelines = {}

for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2": r2_score(y_test, pred)
    })
    pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE")
display(results_df)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for name, pipe in pipelines.items():
    scores = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    cv_results.append({
        "Model": name,
        "CV_RMSE_Mean": scores.mean(),
        "CV_RMSE_Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values("CV_RMSE_Mean")
display(cv_df)

## 5. Evaluation
The final selected model is **Gradient Boosting**, based on the lowest mean 5-fold CV RMSE among the tested models. On the held-out 20% test set it achieved MAE ≈ 1.09, RMSE ≈ 2.26, and R² ≈ 0.80.

In [ ]:
final_model = pipelines["Gradient Boosting"]
test_pred = final_model.predict(X_test)

final_metrics = {
    "MAE": mean_absolute_error(y_test, test_pred),
    "RMSE": mean_squared_error(y_test, test_pred) ** 0.5,
    "R2": r2_score(y_test, test_pred)
}
final_metrics

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test, test_pred, alpha=.75)
mn, mx = min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())
plt.plot([mn,mx],[mn,mx])
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted Selling Price")
plt.show()

## 6. Deployment
The fitted preprocessing + model pipeline is saved as a single artifact so the same transformations are applied during prediction. The Streamlit application in `../app/app.py` loads this artifact and exposes an interactive price estimator.

In [ ]:
joblib.dump(final_model, "../models/best_car_price_model.joblib")
print("Model saved.")

## 7. Limitations and Next Steps
- The dataset is relatively small (299 rows after duplicate removal), so estimates may vary for uncommon vehicle configurations.
- The dataset does not document a real-world valuation date or market geography, so predictions should not be interpreted as current market quotations.
- Additional variables such as condition, service history, location, accident history, and seller-specific information could improve the model.
- A larger and temporally representative dataset should be used before production pricing decisions.